In [1]:
# Intent agent setup: route an analyst question to ONE workspace.
# The analyst never sees or types schema/workspace names - the agent reads
# the catalog descriptions and decides.
import os
import re
import json
import time
from pathlib import Path

import requests
import yaml
from dotenv import load_dotenv

# project root = parent of notebook/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

# opencode inference endpoint (OpenAI-compatible chat completions)
LLM_URL = os.getenv("LLM_URL")
LLM_KEY = os.getenv("SQL_AGENT")     # API key stored under the name SQL_AGENT
LLM_MODEL = os.getenv("LLM_MODEL", "qwen3.8-flash")

missing = [n for n, v in {"LLM_URL": LLM_URL, "SQL_AGENT": LLM_KEY}.items() if not v]
if missing:
    raise ValueError(f"Missing variables in .env: {', '.join(missing)}")

CATALOG_PATH = PROJECT_ROOT / "catalog" / "catalog.yaml"
print(f"model: {LLM_MODEL} | catalog: {CATALOG_PATH.name}")

model: qwen3.8-flash | catalog: catalog.yaml


In [2]:
# llm() is defined in notebook/03_sql_writer.ipynb and exported to src/llm.py
# by that notebook's export cell. Import it instead of copying.
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from src.llm import llm

print("llm imported from src/ (defined first in notebook 03)")

llm imported from src/ (defined first in notebook 03)


In [3]:
# Load the catalog: the intent agent only needs workspace descriptions
# and the policy (the PDP rules matter later, at SQL generation)
catalog = yaml.safe_load(CATALOG_PATH.read_text(encoding="utf-8"))

print("policy:", catalog["policy"])
for name, ws in catalog["workspaces"].items():
    print(f"{name}: {ws['description']}")

policy: {'max_workspaces_per_question': 1, 'cross_schema_joins': 'disallowed', 'cross_schema_fk_count': 0, 'connection_count': 3}
CO: Customer orders, order items, shipments, stores, inventory, products and product reviews
HR: Employees, departments, jobs, locations, regions and job history
SH: Sales facts with product, customer, time, channel and promotion dimensions, plus profit and monthly-sales aggregations


In [4]:
# The intent agent: question -> one or two workspace names (or None).\n# These functions are the first definitions - exported below to src/intent.py.
# TWO workspaces are only legitimate when the catalog records a connection
# between them - the agent sees that list, and the code validates it again
# (never trust the model's routing on its own).
def parse_json_object(text):
    # tolerate fences or extra prose: extract the first {...} block
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

def connected_pair_exists(workspaces, catalog):
    # a two-workspace route is valid only if a catalog connection links them
    wanted = set(workspaces)
    for conn in catalog.get("connections", []):
        sides = {t.split(".", 1)[0] for t in conn["tables"]}
        if sides == wanted:
            return True
    return False

def intent_agent(question, catalog):
    # the prompt template lives INSIDE the function so this cell exports
    # verbatim to src/intent.py (no module-level constants to carry along)
    intent_prompt = """You route analyst questions to the right data workspace.

Workspaces (name: what it contains):
{workspace_list}

Connections between workspaces (pairs that may be combined):
{connection_list}

Rules:
- Pick ONE workspace for a normal question.
- Pick TWO only if the question genuinely needs data from both AND the pair
  is listed above as connected.
- If no workspace can answer the question, pick null.
Answer with ONLY a JSON object, no other text:
{{"workspaces": ["CO"]}}   or   {{"workspaces": ["HR", "SH"]}}   or   {{"workspaces": null}}"""
    workspace_list = "\n".join(
        f"- {name}: {ws['description']}" for name, ws in catalog["workspaces"].items()
    )
    # the catalog policy decides whether two-workspace routing is allowed at
    # all: when cross-schema joins are disabled the model must never see the
    # candidate pairs, so it cannot be tempted into a two-workspace route
    if catalog["policy"].get("cross_schema_joins") == "disallowed":
        connection_list = "- none (cross-schema joins are disabled)"
    else:
        pairs = [
            f"- {' <-> '.join(conn['tables'])} (join key: {conn['join_key']})"
            for conn in catalog.get("connections", [])
        ]
        connection_list = "\n".join(pairs) if pairs else "- none"
    prompt = (
        intent_prompt.format(workspace_list=workspace_list, connection_list=connection_list)
        + f"\n\nQuestion: {question}"
    )
    answer = llm(
        prompt,
        system="You are a routing agent.",
        session="ses_queryllm_intent",
        temperature=0,
        # routing keeps thinking ON: its reasoning chain is short (~1-2s)
        # and measurably improves routing accuracy; the writer/validator
        # run with thinking OFF because their chains cost 40s+
        enable_thinking=True,
    )
    parsed = parse_json_object(answer) or {}
    workspaces = parsed.get("workspaces")

    # normalize + validate: every name must exist in the catalog
    if isinstance(workspaces, str):
        workspaces = [workspaces]
    if not isinstance(workspaces, list):
        workspaces = []
    workspaces = [w for w in workspaces if w in catalog["workspaces"]]
    workspaces = list(dict.fromkeys(workspaces))          # keep order, drop dupes

    # enforce the catalog policy maximum (2 when connections exist)
    max_ws = catalog["policy"].get("max_workspaces_per_question", 1)
    workspaces = workspaces[:max_ws]

    if len(workspaces) == 2 and not connected_pair_exists(workspaces, catalog):
        # never route blind: fall back to the single strongest workspace
        print(f"  note: {workspaces} not connected in catalog -> falling back to first")
        workspaces = workspaces[:1]

    return workspaces or None

In [5]:
# Analyst-eye test questions (business language, no schema names).
# The last one is deliberately outside every workspace: routing must say None.
QUESTIONS = [
    "How much revenue did we make per month?",
    "How many employees are in each department?",
    "Which stores generated the most revenue last year?",
    "Which products sold the most units?",
    "Which stores are low on stock for their products?",
    "How long do shipments usually take to be delivered?",
    "How many servers do we have in the data center?",
    # cross-schema: the HR <-> SH countries pair is connected in the catalog
    "Compare the countries listed in the HR system with the countries where we have sales",
    # cross-schema: the CO <-> SH pair is connected (join key unresolved)
    "Which products do we sell in the sales system that are also in the order catalog?",
]

routes = {}
for question in QUESTIONS:
    workspaces = intent_agent(question, catalog)
    routes[question] = workspaces
    print(f"{question}\n    -> {workspaces}")

How much revenue did we make per month?
    -> ['SH']


How many employees are in each department?
    -> ['HR']


Which stores generated the most revenue last year?
    -> ['CO']


Which products sold the most units?
    -> ['SH']


Which stores are low on stock for their products?
    -> ['CO']


How long do shipments usually take to be delivered?
    -> ['CO']


How many servers do we have in the data center?
    -> None


Compare the countries listed in the HR system with the countries where we have sales
    -> None


Which products do we sell in the sales system that are also in the order catalog?
    -> None


In [6]:
# Summary: routing table + hit rate against the catalog
known = sum(1 for w in routes.values() if w is not None)
for question, workspaces in routes.items():
    if workspaces:
        label = " + ".join(workspaces)
        print(f"{label:<8} <- {question}")
    else:
        print(f"---      <- {question} (outside every workspace)")
print(f"\nrouted: {known}/{len(routes)} questions matched a workspace")

SH       <- How much revenue did we make per month?
HR       <- How many employees are in each department?
CO       <- Which stores generated the most revenue last year?
SH       <- Which products sold the most units?
CO       <- Which stores are low on stock for their products?
CO       <- How long do shipments usually take to be delivered?
---      <- How many servers do we have in the data center? (outside every workspace)
---      <- Compare the countries listed in the HR system with the countries where we have sales (outside every workspace)
---      <- Which products do we sell in the sales system that are also in the order catalog? (outside every workspace)

routed: 6/9 questions matched a workspace


In [7]:
# Export to src/: the functions defined above in this notebook (first
# definitions) are written to src/intent.py so later notebooks import them
# instead of copying. Re-run this notebook after changing them.
import inspect

SRC_DIR = PROJECT_ROOT / "src"
def export_module(filename, header, names):
    parts = [header]
    for name in names:
        parts.append("\n\n" + inspect.getsource(globals()[name]).rstrip() + "\n")
    (SRC_DIR / filename).write_text("".join(parts), encoding="utf-8")
    print(f"exported src/{filename}: {', '.join(names)}")

export_module(
    "intent.py",
    "# Intent agent (first defined in notebook/05_intent_agent.ipynb).\n"
    "# Routes an analyst question to ONE workspace, or TWO only when the\n"
    "# catalog records a connection between the pair. Never trusts the model:\n"
    "# every answer is re-validated against catalog.yaml.\n"
    "import json\nimport re\n\nfrom src.llm import llm\n",
    ["parse_json_object", "connected_pair_exists", "intent_agent"],
)

exported src/intent.py: parse_json_object, connected_pair_exists, intent_agent
